In [1]:
import nibabel as nib
from nibabel.processing import resample_to_output
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from ipywidgets import interact, IntSlider
import numpy as np
from resources.cut_off_edges import cut_off_edges

In [2]:
PROJECT_DIR = Path.cwd().parents[1]

OUTPUTS_DIR = PROJECT_DIR / "outputs"
SCAN_CENTRES_PATH = OUTPUTS_DIR / "scan_centres.csv"
CUT_IMAGES_DIR = PROJECT_DIR / "outputs" / "cut_images"
scan_centres = pd.read_csv(SCAN_CENTRES_PATH)
target_map = scan_centres.set_index("uid")["pathological"]

PROJECT_DIR = Path(r"C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons")
NIFTI_DIR = PROJECT_DIR / "data" / "niftis"

In [3]:
uid_to_show = ["3m9iqbq9","01nouhtc"]

In [4]:
def create_mri_view(uid, volume, diagnosis):

    @interact(
        x=IntSlider(
            min=0,
            max=volume.shape[0] - 1,
            value=volume.shape[0] // 2,
            continuous_update=False
        ),
        y=IntSlider(
            min=0,
            max=volume.shape[1] - 1,
            value=volume.shape[1] // 2,
            continuous_update=False
        ),
        z=IntSlider(
            min=0,
            max=volume.shape[2] - 1,
            value=volume.shape[2] // 2,
            continuous_update=False
        )
    )
    def show_mri(x, y, z):

        fig, ax = plt.subplots(1, 3, figsize=(15, 5))

        ax[0].imshow(
            volume[x, :, :].T,
            cmap="gray",
            origin="lower"
        )

        ax[1].imshow(
            volume[:, y, :].T,
            cmap="gray",
            origin="lower"
        )

        ax[2].imshow(
            volume[:, :, z].T,
            cmap="gray",
            origin="lower"
        )

        ax[0].set_title(f"Oś 0, przekrój {x}")
        ax[1].set_title(f"Oś 1, przekrój {y}")
        ax[2].set_title(f"Oś 2, przekrój {z}")

        ax[0].set_xlabel("Y")
        ax[0].set_ylabel("Z")

        ax[1].set_xlabel("X")
        ax[1].set_ylabel("Z")

        ax[2].set_xlabel("X")
        ax[2].set_ylabel("Y")

        fig.suptitle(
            f"UID: {uid} | {diagnosis} | shape={volume.shape}",
            fontsize=14
        )

        plt.tight_layout()
        plt.show()


for uid in uid_to_show:

    file = CUT_IMAGES_DIR / f"{uid}.nii.gz"

    target = scan_centres.loc[
        scan_centres["uid"] == uid,
        "pathological"
    ].iloc[0]

    diagnosis = (
        "Pathological"
        if target == 1
        else "Not pathological"
    )

    volume = nib.load(file).get_fdata()

    print("Plik:", file.name)
    print("Shape:", volume.shape)
    print("Min:", volume.min())
    print("Max:", volume.max())



    create_mri_view(
        uid=uid,
        volume=volume,
        diagnosis=diagnosis
    )

    image = nib.load(file)
    print("Shape:", image.shape)
    print("Voxel size:", image.header.get_zooms()[:3])
    print("Affine:")
    print(image.affine)

    resampled_image = resample_to_output(
        image,
        voxel_sizes=(2.46, 2.46, 2.46),
        order=1,
    )

    volume_before = image.get_fdata(dtype=np.float32)
    volume_after = resampled_image.get_fdata(dtype=np.float32)

    print("Przed:")
    print("Shape:", volume_before.shape)
    print("Min:", volume_before.min())
    print("Max:", volume_before.max())
    print("Mean:", volume_before.mean())
    print("Std:", volume_before.std())

    print()

    print("Po:")
    print("Shape:", volume_after.shape)
    print("Min:", volume_after.min())
    print("Max:", volume_after.max())
    print("Mean:", volume_after.mean())
    print("Std:", volume_after.std())

    create_mri_view(
        uid=uid,
        volume=volume_after,
        diagnosis=diagnosis
    )


Plik: 3m9iqbq9.nii.gz
Shape: (60, 60, 55)
Min: 0.0
Max: 11646.000027805567


interactive(children=(IntSlider(value=30, continuous_update=False, description='x', max=59), IntSlider(value=3…

Shape: (60, 60, 55)
Voxel size: (np.float32(2.46), np.float32(2.46), np.float32(2.46))
Affine:
[[   2.46000004    0.           -0.          -53.25998688]
 [   0.            2.46000004   -0.          -78.84000397]
 [   0.            0.            2.46000004 -524.22003174]
 [   0.            0.            0.            1.        ]]
Przed:
Shape: (60, 60, 55)
Min: 0.0
Max: 11646.0
Mean: 1581.8906
Std: 1127.0353

Po:
Shape: (61, 61, 56)
Min: 0.0
Max: 11646.0
Mean: 1503.1211
Std: 1151.2424


interactive(children=(IntSlider(value=30, continuous_update=False, description='x', max=60), IntSlider(value=3…

Plik: 01nouhtc.nii.gz
Shape: (100, 100, 90)
Min: 0.0
Max: 7113.9998457580805


interactive(children=(IntSlider(value=50, continuous_update=False, description='x', max=99), IntSlider(value=5…

Shape: (100, 100, 90)
Voxel size: (np.float32(1.46827), np.float32(1.46827), np.float32(1.5))
Affine:
[[ 1.46826994e+00  0.00000000e+00 -0.00000000e+00 -7.56161499e+01]
 [ 0.00000000e+00  1.46826994e+00 -0.00000000e+00  1.22190300e+02]
 [ 0.00000000e+00  0.00000000e+00  1.50000000e+00 -1.18475000e+03]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
Przed:
Shape: (100, 100, 90)
Min: 0.0
Max: 7114.0
Mean: 1197.7079
Std: 642.6233

Po:
Shape: (61, 61, 56)
Min: 0.0
Max: 7003.7866
Mean: 1135.0065
Std: 676.85175


interactive(children=(IntSlider(value=30, continuous_update=False, description='x', max=60), IntSlider(value=3…